# பாடம் 05 - சார்ந்த RAG


## அமைப்பு

இந்த நோட்புக் Microsoft Agent Framework ஐ பயன்படுத்தி Agentic RAG (Retrieval-Augmented Generation) முறைப்பாட்டை விளக்குகிறது.

**முந்தைய தேவைகள்:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — உங்கள் Azure AI Search சேவை முடிவு புள்ளி
- `AZURE_SEARCH_API_KEY` — உங்கள் Azure AI Search API விசை
- சூழல் மாறிகளின் மூலம் அமைக்கப்பட்ட Azure OpenAI நடவடிக்கை
- Azure CLI அங்கீகாரம் பெற்றது (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Agentic RAG என்றால் என்ன?

பாரம்பரிய RAG ஒரு நிலையான பைப்்லைனைப் பின்பற்றுகிறது: ஆவணங்களை மீட்டெடுக்கும், பின்னர் பதிலை உருவாக்குகிறது. **Agentic RAG**கான உதவியாளர் தனது சுயாதீனத்தைக் கொண்டு **எப்போது** மற்றும் **எப்படி** தகவல்களை மீட்டெடுக்க வேண்டும் என்பதைத் தீர்மானிக்க முடியும்.

Agentic RAG உடன், உதவியாளர்:
- ஒரு கேள்விக்கு பதில் அளிக்கும்வரை மீட்டெடுப்பே தேவையா என்பதை **தீர்மானிக்க** முடியும்
- எந்த தரவுத்தளத்தை அல்லது கருவியை விசாரிக்க வேண்டும் என்பதை **தேர்ந்தெடுக்க** முடியும்
- மீட்டெடுக்கப்பட்ட முடிவுகளை **மதிப்பாய்வு செய்து** முதலாவது முயற்சி போதுமானதாக இல்லையெனில் தொடர்ந்த மீட்டெடுப்புகளை செய்யலாம்
- பல மீட்டெடுப்பு படிகளிலிருந்து தகவல்களை **ஒருமித்தமான பதிலாக** இணைக்க முடியும்

இது உதவியாளரை நிலையான மீட்டெடுத்து-பின்பு-உருவாக்கும் பைப்்லைனுடன் ஒப்பிடும்போது அதிகச் சுருக்கமானதும் துல்லியமானதும் ஆகும்.


## ஒரு தேடல் கருவியை உருவாக்கல்

ஏஜென்டிக் RAG-இல், வெளிப்புற தரவு மூலங்கள் **கருவிகள்** என மூடியுள்ளன, அவற்றை ஏஜென்ட் தேவையானபோது அழைக்கலாம். இதனால் ஏஜென்ட், மீட்டெடுப்பை ஒரு கட்டாயக் கட்டமாக அல்லாமல், எடுத்துக்கொள்ளக்கூடிய மற்ற செயல்பாடுகளில் ஒன்றாகக் கருதுகிறது.

கீழே, ஒரு பயண அறிவுத்தளம் உருவாக்கி, அதனை ஏஜென்ட் அழைக்கக்கூடிய கருவியாக வெளிப்படுத்துகிறோம், ஏனெனில் பயண இடத் தகவல்களை அறிவதற்கு.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG முகவரியை உருவாக்குதல்

இப்போது நாம் ஒரு முகவரியை உருவாக்குகிறோம், இது **பதில் அளிப்பதற்கு முன்பு எப்போதும் தகவலை மீட்டெடுக்க உத்தரவிடப்பட்டுள்ளது**. முகவரி தன் பதில்களை தன்னுடைய பயிற்சி தரவுக்கு பதிலாக அறிவுத்தளத்தில் அடிப்படையாக்க `search_travel_knowledge` கருவியை பயன்படுத்துகிறது.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## அடிபடையாக மீட்பு — மேக்கர்-செக்கர் முறை

Agentic RAG இன் முக்கியமான நன்மை **அடிபடையாக மீட்பு** ஆகும். முகவர் தன் ஆரம்ப கண்டுபிடிப்புகளை சரிபார்க்க, நுட்பப்படுத்த, அல்லது விரிவுபடுத்த பல சுற்றுகளாக தேடல் செய்ய முடியும் — "மேக்கர்-செக்கர்" வேலைப்பாடை போல:

1. **மேக்கர் படி**: முகவர் ஆரம்ப தகவலை மீட்டெடுத்து ஒரு பதிலை முன்மொழிக்கிறார்.
2. **செக்கர் படி**: முகவர் விவரங்களை சரிபார்க்க அல்லது காலியிடங்களை நிரப்ப கூடுதல் மீட்புகளை செய்கிறார்.

கீழே, முகவரிடம் பல இடங்களை ஒப்பிட வேண்டிய கேள்வி கேட்கப்படுகிறது, இது பலமுறை தேடலை தூண்டும்.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## சுருக்கம்

இந்த பாடத்தில், Microsoft Agent Framework-ஐ பயன்படுத்தி ஒரு **Agentic RAG** கணினி அமைப்பை உருவாக்குவது எப்படி என்பதை நீங்கள் கற்றுக்கொண்டீர்கள்:

- **Agentic RAG** முகவர்கள் எப்போது தகவலை பெறவேண்டும் என்பதை தன்னிச்சையாக தீர்மானிக்கலாம், இதனால் தகவல் தேடல் நிலையானதைவிட இயக்கத்துடனாக இருக்கும்.
- **கருவிகள் தரவுத் தளங்களாக**: உள் அறிவுத்தளங்களை (Azure AI Search போன்றவை) முகவர் அழைக்கக்கூடிய கருவிகளாக மாற்றி அமைத்துள்ளோம்.
- **மீண்டும் மீண்டும் பெறுதல்**: maker-checker முறையால் முகவர் பல முறை தகவல் தேடல், சரிபார்த்தல் மற்றும் தூய்மை செய்யும் சுற்றுகளை செயல்படுத்தி இறுதி பதிலை வழங்க முடியும்.

உற்பத்தியில், நினைவகத்தில் உள்ள `TRAVEL_KNOWLEDGE_BASE` ஐ பெரிய அளவிலான பயண ஆவணத் தேடலுக்கு Azure AI Search குறியீட்டுடன் மாற்றுவீர்கள்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
